## Data cleaning

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

RAW_PATH = "../logs.parquet"

df_raw = pd.read_parquet(RAW_PATH)
print("Raw data shape:", df_raw.shape)


Raw data shape: (13352, 26)


In [2]:
df = df_raw.copy()


rename_map = {
    "session_id": "game_session_id",
    "player_mark": "played_by",
    "timestamp": "logged_at",
    "final_result": "outcome",
}
df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})

print("Columns after rename:", df.columns.tolist())
# 2.1 logged_at -> datetime (if present)
if "logged_at" in df.columns:
    df["logged_at"] = pd.to_datetime(df["logged_at"], errors="coerce")

# 2.2 MCTS-related numeric columns – enforce numeric dtype
numeric_cols = [
    "mcts_iterations",
    "exploration_constant",
    "best_visit_count",
    "best_q_value",
    "move_confidence",
    "branching_factor",
    "explored_actions",
    "mcts_tree_size",
    "rollout_count",
    "rollout_depth_avg",
    "rollout_depth_max",
    "mcts_search_duration_ms",
    "ai_latency_ms",
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

print("Dtypes after basic cleaning:")
print(df.dtypes)


Columns after rename: ['id', 'game_session_id', 'played_by', 'board', 'legal_moves', 'ai_move', 'game_status', 'logged_at', 'mcts_iterations', 'exploration_constant', 'visit_counts_json', 'action_scores_json', 'policy_vector_json', 'best_visit_count', 'best_q_value', 'move_confidence', 'branching_factor', 'explored_actions', 'mcts_tree_size', 'rollout_count', 'rollout_depth_avg', 'rollout_depth_max', 'mcts_search_duration_ms', 'ai_latency_ms', 'ai_difficulty', 'outcome']
Dtypes after basic cleaning:
id                                 object
game_session_id                    object
played_by                          object
board                              object
legal_moves                        object
ai_move                            object
game_status                        object
logged_at                  datetime64[ns]
mcts_iterations                     int64
exploration_constant              float64
visit_counts_json                  object
action_scores_json               

In [3]:
sort_cols = ["game_session_id"]
if "logged_at" in df.columns:
    sort_cols.append("logged_at")

df = df.sort_values(sort_cols).reset_index(drop=True)

# Create move_index per game
df["move_index"] = df.groupby("game_session_id").cumcount() + 1

print("move_index summary:")
print(df["move_index"].describe())


move_index summary:
count    13352.00000
mean         4.12867
std          2.20551
min          1.00000
25%          2.00000
50%          4.00000
75%          6.00000
max          9.00000
Name: move_index, dtype: float64


In [4]:
before = df.shape[0]

if "logged_at" in df.columns:
    df = df[df["logged_at"].notna()].copy()

# if "outcome" in df.columns:
#     df = df[df["outcome"].notna()].copy()

after = df.shape[0]
print(f"Dropped {before - after} rows with NaN logged_at.")
print("Shape after row cleaning:", df.shape)

if "outcome" in df.columns:
    print("\nOutcome distribution after row cleaning:")
    print(df["outcome"].value_counts(dropna=False))

print("\nMissing values per column (after row cleaning):")
display(df.isna().sum())


Dropped 0 rows with NaN logged_at.
Shape after row cleaning: (13352, 27)

Outcome distribution after row cleaning:
outcome
None    11452
WIN      1447
DRAW      453
Name: count, dtype: int64

Missing values per column (after row cleaning):


id                             0
game_session_id                0
played_by                      0
board                          0
legal_moves                    0
ai_move                        0
game_status                    0
logged_at                      0
mcts_iterations                0
exploration_constant           0
visit_counts_json              0
action_scores_json             0
policy_vector_json             0
best_visit_count               0
best_q_value                   0
move_confidence                0
branching_factor               0
explored_actions               0
mcts_tree_size                 0
rollout_count                  0
rollout_depth_avg              0
rollout_depth_max              0
mcts_search_duration_ms        0
ai_latency_ms                  0
ai_difficulty                  0
outcome                    11452
move_index                     0
dtype: int64

In [5]:
before = df.shape[0]
df = df.drop_duplicates()
after = df.shape[0]

print(f"Dropped {before - after} exact duplicate rows.")
print("Shape after deduplication:", df.shape)


Dropped 0 exact duplicate rows.
Shape after deduplication: (13352, 27)


In [6]:
print("Final missing values per column:")
display(df.isna().sum())

if "outcome" in df.columns:
    print("\nFinal outcome distribution:")
    print(df["outcome"].value_counts(dropna=False))

print("\nFinal move_index summary:")
print(df["move_index"].describe())

display(df.head())


Final missing values per column:


id                             0
game_session_id                0
played_by                      0
board                          0
legal_moves                    0
ai_move                        0
game_status                    0
logged_at                      0
mcts_iterations                0
exploration_constant           0
visit_counts_json              0
action_scores_json             0
policy_vector_json             0
best_visit_count               0
best_q_value                   0
move_confidence                0
branching_factor               0
explored_actions               0
mcts_tree_size                 0
rollout_count                  0
rollout_depth_avg              0
rollout_depth_max              0
mcts_search_duration_ms        0
ai_latency_ms                  0
ai_difficulty                  0
outcome                    11452
move_index                     0
dtype: int64


Final outcome distribution:
outcome
None    11452
WIN      1447
DRAW      453
Name: count, dtype: int64

Final move_index summary:
count    13352.00000
mean         4.12867
std          2.20551
min          1.00000
25%          2.00000
50%          4.00000
75%          6.00000
max          9.00000
Name: move_index, dtype: float64


,id,game_session_id,played_by,board,legal_moves,ai_move,game_status,logged_at,mcts_iterations,exploration_constant,visit_counts_json,action_scores_json,policy_vector_json,best_visit_count,best_q_value,move_confidence,branching_factor,explored_actions,mcts_tree_size,rollout_count,rollout_depth_avg,rollout_depth_max,mcts_search_duration_ms,ai_latency_ms,ai_difficulty,outcome,move_index
0,01eb4219-2253-4f64-84fa-e53cecab2315,0043a101-b212-405a-88d1-f7cc9e8be0b8,X,"[["""", """", """"], ["""", """", """"], ["""", """", """"]]","[{""row"": 0, ""col"": 0}, {""row"": 0, ""col"": 1}, {...","{""row"": 0, ""col"": 0}",IN_PROGRESS,2025-12-16 11:33:46.986500,5,0.707107,"{'0,0': 1, '0,1': 1, '0,2': 1, '1,0': 1, '1,1'...","{'0,0': 1.0, '0,1': 1.0, '0,2': 1.0, '1,0': 0....","{'0,0': 0.2, '0,1': 0.2, '0,2': 0.2, '1,0': 0....",1,1.000000,0.20,9,5,6,5,0,0,5,5,EASY,None,1
1,8f62b4ac-fee2-4e04-bcf1-e3a99b73ec64,0043a101-b212-405a-88d1-f7cc9e8be0b8,O,"[[""X"", """", """"], ["""", """", """"], ["""", """", """"]]","[{""row"": 0, ""col"": 1}, {""row"": 0, ""col"": 2}, {...","{""row"": 1, ""col"": 1}",IN_PROGRESS,2025-12-16 11:33:47.015392,50,0.707107,"{'0,1': 2, '0,2': 2, '1,0': 2, '1,1': 24, '1,2...","{'0,1': -1.0, '0,2': -1.0, '1,0': -1.0, '1,1':...","{'0,1': 0.04, '0,2': 0.04, '1,0': 0.04, '1,1':...",24,0.208333,0.48,8,8,51,50,0,0,22,22,HARD,None,2
2,e55ca37d-d4ae-4753-b634-099ff211e92e,0043a101-b212-405a-88d1-f7cc9e8be0b8,X,"[[""X"", """", """"], ["""", ""O"", """"], ["""", """", """"]]","[{""row"": 0, ""col"": 1}, {""row"": 0, ""col"": 2}, {...","{""row"": 0, ""col"": 1}",IN_PROGRESS,2025-12-16 11:33:47.023794,5,0.707107,"{'0,1': 1, '0,2': 1, '1,0': 1, '1,2': 1, '2,0'...","{'0,1': 1.0, '0,2': 1.0, '1,0': 1.0, '1,2': -1...","{'0,1': 0.2, '0,2': 0.2, '1,0': 0.2, '1,2': 0....",1,1.000000,0.20,7,5,6,5,0,0,1,1,EASY,None,3
3,f1137609-6e74-447b-bb0a-2ea967b7ec37,0043a101-b212-405a-88d1-f7cc9e8be0b8,O,"[[""X"", ""X"", """"], ["""", ""O"", """"], ["""", """", """"]]","[{""row"": 0, ""col"": 2}, {""row"": 1, ""col"": 0}, {...","{""row"": 0, ""col"": 2}",IN_PROGRESS,2025-12-16 11:33:47.036333,50,0.707107,"{'0,2': 45, '1,0': 1, '1,2': 1, '2,0': 1, '2,1...","{'0,2': 0.8444444444444444, '1,0': -1.0, '1,2'...","{'0,2': 0.9, '1,0': 0.02, '1,2': 0.02, '2,0': ...",45,0.844444,0.90,6,6,45,50,0,0,6,6,HARD,None,4
4,372642ca-dadc-41fa-a19f-11aaccb21d62,0043a101-b212-405a-88d1-f7cc9e8be0b8,X,"[[""X"", ""X"", ""O""], ["""", ""O"", """"], ["""", """", """"]]","[{""row"": 1, ""col"": 0}, {""row"": 1, ""col"": 2}, {...","{""row"": 1, ""col"": 0}",IN_PROGRESS,2025-12-16 11:33:47.043378,5,0.707107,"{'1,0': 1, '1,2': 1, '2,0': 1, '2,1': 1, '2,2'...","{'1,0': -1.0, '1,2': -1.0, '2,0': -1.0, '2,1':...","{'1,0': 0.2, '1,2': 0.2, '2,0': 0.2, '2,1': 0....",1,-1.000000,0.20,5,5,6,5,0,0,0,0,EASY,None,5


In [7]:
CLEAN_PATH = "../logs_clean_base.parquet"
df.to_parquet(CLEAN_PATH)

print(f"Saved cleaned base dataset to: {CLEAN_PATH}")
print("Final cleaned shape:", df.shape)


Saved cleaned base dataset to: ../logs_clean_base.parquet
Final cleaned shape: (13352, 27)
